# YouTube Experiment Runner

**Purpose**: Automatically validate GACS using real YouTube algorithm metrics.

## Pipeline Overview
1. Authenticate YouTube Data API (OAuth2)
2. Generate metadata (title, description, tags) using Claude
3. Upload videos alternating gacs/baseline
4. Collect metrics every 24 hours
5. Generate statistical comparison report

## Prerequisites
- Run `gacs_video_generator.ipynb` first
- Set up Google Cloud OAuth credentials (see Section 1)

## 1. YouTube API Setup Guide

### Step 1: Create a Google Cloud Project
1. Go to [Google Cloud Console](https://console.cloud.google.com/)
2. Click "Select a project" → "New Project"
3. Enter project name (e.g., "GACS Research")
4. Click "Create"

### Step 2: Enable YouTube Data API v3
1. In the Cloud Console, go to "APIs & Services" → "Library"
2. Search for "YouTube Data API v3"
3. Click on it and press "Enable"

### Step 3: Enable YouTube Analytics API
1. Search for "YouTube Analytics API"
2. Click on it and press "Enable"

### Step 4: Configure OAuth Consent Screen
1. Go to "APIs & Services" → "OAuth consent screen"
2. Select "External" (unless you have a Workspace account)
3. Fill in required fields:
   - App name: "GACS Research"
   - User support email: your email
   - Developer contact: your email
4. Click "Save and Continue"
5. Add scopes:
   - `https://www.googleapis.com/auth/youtube.upload`
   - `https://www.googleapis.com/auth/youtube.readonly`
   - `https://www.googleapis.com/auth/yt-analytics.readonly`
6. Add yourself as a test user

### Step 5: Create OAuth 2.0 Credentials
1. Go to "APIs & Services" → "Credentials"
2. Click "Create Credentials" → "OAuth client ID"
3. Application type: "Desktop app"
4. Name: "GACS Desktop Client"
5. Click "Create"
6. Download the JSON file
7. Rename it to `client_secrets.json` and place it in the project root

### Step 6: First-time Authentication
- Running this notebook will open a browser for OAuth authorization
- Sign in with your Google account
- Grant the requested permissions
- Credentials will be cached in `youtube_credentials.json`

## 2. Setup & Configuration

In [ ]:
# Install dependencies (run once)
# !pip install google-api-python-client google-auth-oauthlib google-auth-httplib2 pandas matplotlib seaborn scipy anthropic tqdm fpdf

In [ ]:
import json
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import Dict, List, Optional

import anthropic
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaFileUpload
from scipy import stats
from tqdm.notebook import tqdm

from gacs_config import CLAUDE_MODEL, EXPERIMENTS_DIR, GENERATED_DIR, PROJECT_ROOT, setup_logging, validate_config

logger = setup_logging("youtube_experiment_runner")

print("All dependencies loaded successfully!")

In [ ]:
# Configuration
BASE_DIR = PROJECT_ROOT
GACS_DIR = GENERATED_DIR / "gacs"
BASELINE_DIR = GENERATED_DIR / "baseline"

# OAuth settings
CLIENT_SECRETS_FILE = BASE_DIR / "client_secrets.json"
CREDENTIALS_FILE = BASE_DIR / "youtube_credentials.json"

# Validate configuration
validate_config()

print(f"Base directory: {BASE_DIR}")
print(f"Experiments directory: {EXPERIMENTS_DIR}")
print(f"Client secrets exists: {CLIENT_SECRETS_FILE.exists()}")

## 3. Authentication

In [ ]:
def get_authenticated_services():
    """
    Authenticate and return YouTube API services.
    
    Returns:
        Tuple of (youtube_service, youtube_analytics_service)
    """
    credentials = None

    # Check for existing credentials
    if CREDENTIALS_FILE.exists():
        with open(CREDENTIALS_FILE, 'r') as f:
            creds_data = json.load(f)
        credentials = Credentials.from_authorized_user_info(creds_data, SCOPES)

    # Refresh or get new credentials
    if not credentials or not credentials.valid:
        if credentials and credentials.expired and credentials.refresh_token:
            print("Refreshing expired credentials...")
            credentials.refresh(Request())
        else:
            if not CLIENT_SECRETS_FILE.exists():
                raise FileNotFoundError(
                    f"client_secrets.json not found at {CLIENT_SECRETS_FILE}\n"
                    "Please follow the setup guide in Section 1."
                )

            print("Starting OAuth flow...")
            print("A browser window will open for authentication.")

            flow = InstalledAppFlow.from_client_secrets_file(
                str(CLIENT_SECRETS_FILE), SCOPES
            )
            credentials = flow.run_local_server(port=8080)

        # Save credentials
        with open(CREDENTIALS_FILE, 'w') as f:
            f.write(credentials.to_json())
        print(f"Credentials saved to {CREDENTIALS_FILE}")

    # Build services
    youtube = build('youtube', 'v3', credentials=credentials)
    youtube_analytics = build('youtubeAnalytics', 'v2', credentials=credentials)

    print("Successfully authenticated with YouTube API!")
    return youtube, youtube_analytics

# Uncomment to authenticate
# youtube, youtube_analytics = get_authenticated_services()

## 4. Metadata Generation Prompts

In [ ]:
# Initialize Claude client
client = anthropic.Anthropic()

TITLE_PROMPT = """Generate a YouTube title.
Under 60 characters.
Emotional but not clickbait.
No emojis.

Context:
- Video mood: {mood_words}
- Video group: {group}

Respond with ONLY the title, nothing else."""

DESCRIPTION_PROMPT = """Generate a short YouTube description.
Max 2 sentences.
No hashtags.
No marketing language.

Context:
- Video mood: {mood_words}
- Video group: {group}

Respond with ONLY the description, nothing else."""

TAGS_PROMPT = """Generate 8 to 12 single-word tags based on mood and style.

Context:
- Video mood: {mood_words}
- Video style: {style_words}

Rules:
- Single words only
- No phrases
- Relevant to content

Respond with ONLY comma-separated tags, nothing else.
Example: emotional, cinematic, dramatic, powerful, inspiring, music, film, mood"""

print("Metadata generation prompts defined.")

In [ ]:
def generate_video_metadata(video_metadata: Dict) -> Dict:
    """
    Generate YouTube metadata (title, description, tags) using Claude.
    
    Args:
        video_metadata: Metadata dict from generated video
    
    Returns:
        Dict with title, description, tags
    """
    group = video_metadata.get('group', 'unknown')
    mood_words = video_metadata.get('mood_words', [])
    mood_str = ', '.join(mood_words) if mood_words else 'varied'
    style_str = 'cinematic, mixed'  # Default if not available

    # Generate title
    title_prompt = TITLE_PROMPT.format(mood_words=mood_str, group=group)
    try:
        response = client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=64,
            messages=[{"role": "user", "content": title_prompt}]
        )
        title = response.content[0].text.strip()[:60]  # Enforce limit
    except Exception as e:
        print(f"Error generating title: {e}")
        title = f"GACS Video - {group.capitalize()}"

    time.sleep(0.5)  # Rate limiting

    # Generate description
    desc_prompt = DESCRIPTION_PROMPT.format(mood_words=mood_str, group=group)
    try:
        response = client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=128,
            messages=[{"role": "user", "content": desc_prompt}]
        )
        description = response.content[0].text.strip()
    except Exception as e:
        print(f"Error generating description: {e}")
        description = "A video created for research purposes."

    time.sleep(0.5)

    # Generate tags
    tags_prompt = TAGS_PROMPT.format(mood_words=mood_str, style_words=style_str)
    try:
        response = client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=128,
            messages=[{"role": "user", "content": tags_prompt}]
        )
        tags_text = response.content[0].text.strip()
        tags = [t.strip() for t in tags_text.split(',')][:12]  # Max 12 tags
    except Exception as e:
        print(f"Error generating tags: {e}")
        tags = ['video', 'research', 'mood', 'emotional']

    return {
        'title': title,
        'description': description,
        'tags': tags
    }

print("Metadata generation function defined.")

## 5. Video Upload

In [ ]:
def upload_video(youtube_service,
                 video_path: Path,
                 title: str,
                 description: str,
                 tags: List[str],
                 category_id: str = "22",  # People & Blogs
                 privacy: str = "unlisted") -> Optional[str]:
    """
    Upload a video to YouTube.
    
    Args:
        youtube_service: Authenticated YouTube API service
        video_path: Path to the video file
        title: Video title
        description: Video description
        tags: List of tags
        category_id: YouTube category ID
        privacy: Privacy status (public, unlisted, private)
    
    Returns:
        YouTube video ID or None if failed
    """
    if not video_path.exists():
        print(f"Error: Video not found at {video_path}")
        return None

    body = {
        'snippet': {
            'title': title,
            'description': description,
            'tags': tags,
            'categoryId': category_id
        },
        'status': {
            'privacyStatus': privacy,
            'selfDeclaredMadeForKids': False
        }
    }

    media = MediaFileUpload(
        str(video_path),
        mimetype='video/mp4',
        resumable=True,
        chunksize=1024*1024
    )

    try:
        print(f"Uploading: {title}")
        print(f"File: {video_path}")

        request = youtube_service.videos().insert(
            part='snippet,status',
            body=body,
            media_body=media
        )

        response = None
        while response is None:
            status, response = request.next_chunk()
            if status:
                print(f"Upload progress: {int(status.progress() * 100)}%")

        video_id = response['id']
        print(f"Upload complete! Video ID: {video_id}")
        print(f"URL: https://www.youtube.com/watch?v={video_id}")

        return video_id

    except HttpError as e:
        print(f"HTTP Error: {e.resp.status} - {e.content}")
        return None
    except Exception as e:
        print(f"Error uploading video: {e}")
        return None

print("Video upload function defined.")

In [ ]:
def upload_all_videos(youtube_service, alternate: bool = True) -> pd.DataFrame:
    """
    Upload all generated videos, alternating GACS and baseline.
    
    Args:
        youtube_service: Authenticated YouTube API service
        alternate: If True, alternate between GACS and baseline uploads
    
    Returns:
        DataFrame with upload results
    """
    # Load all video metadata
    gacs_videos = []
    baseline_videos = []

    # Load GACS videos
    for video_dir in GACS_DIR.iterdir():
        if video_dir.is_dir():
            metadata_path = video_dir / "metadata.json"
            video_path = video_dir / "video.mp4"
            if metadata_path.exists() and video_path.exists():
                with open(metadata_path, 'r') as f:
                    meta = json.load(f)
                meta['video_path'] = str(video_path)
                gacs_videos.append(meta)

    # Load baseline videos
    for video_dir in BASELINE_DIR.iterdir():
        if video_dir.is_dir():
            metadata_path = video_dir / "metadata.json"
            video_path = video_dir / "video.mp4"
            if metadata_path.exists() and video_path.exists():
                with open(metadata_path, 'r') as f:
                    meta = json.load(f)
                meta['video_path'] = str(video_path)
                baseline_videos.append(meta)

    print(f"Found {len(gacs_videos)} GACS videos and {len(baseline_videos)} baseline videos")

    # Interleave videos if alternating
    if alternate:
        videos_to_upload = []
        for i in range(max(len(gacs_videos), len(baseline_videos))):
            if i < len(gacs_videos):
                videos_to_upload.append(gacs_videos[i])
            if i < len(baseline_videos):
                videos_to_upload.append(baseline_videos[i])
    else:
        videos_to_upload = gacs_videos + baseline_videos

    # Upload each video
    results = []
    youtube_map = []

    for video_meta in tqdm(videos_to_upload, desc="Uploading videos"):
        # Generate YouTube metadata
        yt_meta = generate_video_metadata(video_meta)

        # Upload
        video_path = Path(video_meta['video_path'])
        youtube_id = upload_video(
            youtube_service,
            video_path,
            yt_meta['title'],
            yt_meta['description'],
            yt_meta['tags']
        )

        result = {
            'local_video_id': video_meta['video_id'],
            'youtube_id': youtube_id,
            'group': video_meta['group'],
            'title': yt_meta['title'],
            'uploaded_at': datetime.now().isoformat(),
            'success': youtube_id is not None
        }
        results.append(result)

        if youtube_id:
            youtube_map.append({
                'youtube_id': youtube_id,
                'local_video_id': video_meta['video_id'],
                'group': video_meta['group']
            })

        # Wait between uploads
        time.sleep(5)

    # Save YouTube mapping
    map_path = EXPERIMENTS_DIR / "youtube_map.json"
    with open(map_path, 'w') as f:
        json.dump(youtube_map, f, indent=2)
    print(f"\nYouTube mapping saved to: {map_path}")

    return pd.DataFrame(results)

print("Batch upload function defined.")

## 6. Metrics Collection

In [ ]:
def collect_video_metrics(youtube_service, youtube_analytics_service,
                          video_id: str, days_back: int = 7) -> Dict:
    """
    Collect metrics for a single video.
    
    Args:
        youtube_service: YouTube Data API service
        youtube_analytics_service: YouTube Analytics API service
        video_id: YouTube video ID
        days_back: Number of days to look back
    
    Returns:
        Dict with metrics
    """
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=days_back)).strftime('%Y-%m-%d')

    metrics = {
        'video_id': video_id,
        'collected_at': datetime.now().isoformat(),
        'date_range': {'start': start_date, 'end': end_date}
    }

    # Get basic statistics from YouTube Data API
    try:
        response = youtube_service.videos().list(
            part='statistics,contentDetails',
            id=video_id
        ).execute()

        if response['items']:
            stats = response['items'][0]['statistics']
            metrics['views'] = int(stats.get('viewCount', 0))
            metrics['likes'] = int(stats.get('likeCount', 0))
            metrics['comments'] = int(stats.get('commentCount', 0))
    except HttpError as e:
        print(f"Error fetching statistics for {video_id}: {e}")
        metrics['views'] = 0
        metrics['likes'] = 0
        metrics['comments'] = 0

    # Get detailed analytics
    try:
        response = youtube_analytics_service.reports().query(
            ids='channel==MINE',
            startDate=start_date,
            endDate=end_date,
            metrics='views,estimatedMinutesWatched,averageViewDuration,averageViewPercentage,'
                   'subscribersGained,subscribersLost,shares',
            filters=f'video=={video_id}'
        ).execute()

        if response.get('rows'):
            row = response['rows'][0]
            metrics['impressions'] = row[0] if len(row) > 0 else 0
            metrics['watchTime'] = row[1] if len(row) > 1 else 0
            metrics['averageViewDuration'] = row[2] if len(row) > 2 else 0
            metrics['averageViewPercentage'] = row[3] if len(row) > 3 else 0
            metrics['subscribersGained'] = row[4] if len(row) > 4 else 0
            metrics['subscribersLost'] = row[5] if len(row) > 5 else 0
            metrics['shares'] = row[6] if len(row) > 6 else 0
        else:
            # Set defaults
            metrics['impressions'] = 0
            metrics['watchTime'] = 0
            metrics['averageViewDuration'] = 0
            metrics['averageViewPercentage'] = 0
            metrics['subscribersGained'] = 0
            metrics['subscribersLost'] = 0
            metrics['shares'] = 0

    except HttpError as e:
        print(f"Error fetching analytics for {video_id}: {e}")

    # Get CTR if available
    try:
        response = youtube_analytics_service.reports().query(
            ids='channel==MINE',
            startDate=start_date,
            endDate=end_date,
            metrics='impressions,impressionClickThroughRate',
            filters=f'video=={video_id}'
        ).execute()

        if response.get('rows'):
            row = response['rows'][0]
            metrics['impressions'] = row[0] if len(row) > 0 else 0
            metrics['ctr'] = row[1] if len(row) > 1 else 0
        else:
            metrics['ctr'] = 0
    except:
        metrics['ctr'] = 0

    return metrics

print("Metrics collection function defined.")

In [ ]:
def collect_all_metrics(youtube_service, youtube_analytics_service) -> pd.DataFrame:
    """
    Collect metrics for all uploaded videos.
    
    Returns:
        DataFrame with metrics
    """
    # Load YouTube mapping
    map_path = EXPERIMENTS_DIR / "youtube_map.json"
    if not map_path.exists():
        print("No YouTube mapping found. Upload videos first.")
        return pd.DataFrame()

    with open(map_path, 'r') as f:
        youtube_map = json.load(f)

    print(f"Collecting metrics for {len(youtube_map)} videos...")

    all_metrics = []
    for entry in tqdm(youtube_map, desc="Collecting metrics"):
        metrics = collect_video_metrics(
            youtube_service,
            youtube_analytics_service,
            entry['youtube_id']
        )
        metrics['local_video_id'] = entry['local_video_id']
        metrics['group'] = entry['group']
        all_metrics.append(metrics)

        time.sleep(0.5)  # Rate limiting

    # Convert to DataFrame
    df = pd.DataFrame(all_metrics)

    # Save metrics
    metrics_path = EXPERIMENTS_DIR / "metrics.csv"

    # Append to existing or create new
    if metrics_path.exists():
        existing = pd.read_csv(metrics_path)
        df = pd.concat([existing, df], ignore_index=True)

    df.to_csv(metrics_path, index=False)
    print(f"\nMetrics saved to: {metrics_path}")

    return df

print("Batch metrics collection function defined.")

## 7. Statistical Analysis

In [ ]:
def analyze_experiment_results(metrics_df: pd.DataFrame) -> Dict:
    """
    Perform statistical analysis comparing GACS vs baseline.
    
    Args:
        metrics_df: DataFrame with metrics for all videos
    
    Returns:
        Dict with analysis results
    """
    # Get latest metrics for each video
    latest = metrics_df.sort_values('collected_at').groupby('video_id').last().reset_index()

    # Split by group
    gacs = latest[latest['group'] == 'gacs']
    baseline = latest[latest['group'] == 'baseline']

    results = {
        'sample_sizes': {
            'gacs': len(gacs),
            'baseline': len(baseline)
        },
        'metrics': {}
    }

    # Metrics to compare
    metrics_to_compare = [
        'views', 'likes', 'comments', 'watchTime',
        'averageViewDuration', 'averageViewPercentage', 'ctr', 'shares'
    ]

    for metric in metrics_to_compare:
        if metric not in gacs.columns or metric not in baseline.columns:
            continue

        gacs_vals = gacs[metric].dropna().astype(float)
        baseline_vals = baseline[metric].dropna().astype(float)

        if len(gacs_vals) < 2 or len(baseline_vals) < 2:
            continue

        # Calculate statistics
        gacs_mean = gacs_vals.mean()
        baseline_mean = baseline_vals.mean()

        # T-test
        t_stat, p_value = stats.ttest_ind(gacs_vals, baseline_vals)

        # Effect size (Cohen's d)
        pooled_std = np.sqrt(((len(gacs_vals)-1)*gacs_vals.std()**2 +
                              (len(baseline_vals)-1)*baseline_vals.std()**2) /
                             (len(gacs_vals)+len(baseline_vals)-2))
        cohens_d = (gacs_mean - baseline_mean) / pooled_std if pooled_std > 0 else 0

        # Lift percentage
        lift = ((gacs_mean - baseline_mean) / baseline_mean * 100) if baseline_mean > 0 else 0

        results['metrics'][metric] = {
            'gacs_mean': gacs_mean,
            'baseline_mean': baseline_mean,
            'lift_percent': lift,
            't_statistic': t_stat,
            'p_value': p_value,
            'cohens_d': cohens_d,
            'significant': p_value < 0.05
        }

    return results

def print_analysis_report(results: Dict):
    """
    Print formatted analysis report.
    """
    print("\n" + "="*80)
    print("GACS vs Baseline Statistical Analysis")
    print("="*80)
    print(f"\nSample sizes: GACS={results['sample_sizes']['gacs']}, "
          f"Baseline={results['sample_sizes']['baseline']}")

    print(f"\n{'Metric':<25} {'GACS Mean':>12} {'Baseline':>12} {'Lift':>10} "
          f"{'p-value':>10} {'Sig':>5}")
    print("-"*80)

    for metric, data in results['metrics'].items():
        sig = "*" if data['significant'] else ""
        print(f"{metric:<25} {data['gacs_mean']:>12.2f} {data['baseline_mean']:>12.2f} "
              f"{data['lift_percent']:>9.1f}% {data['p_value']:>10.4f} {sig:>5}")

    print("-"*80)
    print("* indicates statistical significance (p < 0.05)")

print("Statistical analysis functions defined.")

## 8. Visualization

In [ ]:
def create_comparison_plots(metrics_df: pd.DataFrame, output_dir: Path = None):
    """
    Create comparison visualizations.
    """
    if output_dir is None:
        output_dir = EXPERIMENTS_DIR

    # Get latest metrics per video
    latest = metrics_df.sort_values('collected_at').groupby('video_id').last().reset_index()

    # Set style
    sns.set_style("whitegrid")

    # Create figure with subplots
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    metrics_to_plot = [
        ('views', 'Views'),
        ('likes', 'Likes'),
        ('watchTime', 'Watch Time (min)'),
        ('averageViewDuration', 'Avg View Duration (s)'),
        ('averageViewPercentage', 'Avg View %'),
        ('ctr', 'CTR (%)')
    ]

    for idx, (metric, label) in enumerate(metrics_to_plot):
        ax = axes[idx // 3, idx % 3]

        if metric in latest.columns:
            sns.boxplot(data=latest, x='group', y=metric, ax=ax, palette=['#2ecc71', '#3498db'])
            ax.set_title(label)
            ax.set_xlabel('')
            ax.set_ylabel(label)
        else:
            ax.text(0.5, 0.5, f'{label}\nNo data', ha='center', va='center')
            ax.set_title(label)

    plt.suptitle('GACS vs Baseline Comparison', fontsize=14, fontweight='bold')
    plt.tight_layout()

    plot_path = output_dir / 'comparison_boxplots.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"Comparison plots saved to: {plot_path}")
    plt.show()

    # Create bar chart with means
    fig, ax = plt.subplots(figsize=(12, 6))

    gacs = latest[latest['group'] == 'gacs']
    baseline = latest[latest['group'] == 'baseline']

    metrics_for_bars = ['views', 'likes', 'watchTime', 'averageViewPercentage']
    existing_metrics = [m for m in metrics_for_bars if m in latest.columns]

    if existing_metrics:
        x = np.arange(len(existing_metrics))
        width = 0.35

        gacs_means = [gacs[m].mean() for m in existing_metrics]
        baseline_means = [baseline[m].mean() for m in existing_metrics]

        bars1 = ax.bar(x - width/2, gacs_means, width, label='GACS', color='#2ecc71')
        bars2 = ax.bar(x + width/2, baseline_means, width, label='Baseline', color='#3498db')

        ax.set_ylabel('Mean Value')
        ax.set_title('Mean Metrics by Group')
        ax.set_xticks(x)
        ax.set_xticklabels(existing_metrics)
        ax.legend()

        # Add value labels
        for bar, val in zip(bars1, gacs_means):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{val:.1f}',
                   ha='center', va='bottom', fontsize=8)
        for bar, val in zip(bars2, baseline_means):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{val:.1f}',
                   ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    bar_path = output_dir / 'comparison_bars.png'
    plt.savefig(bar_path, dpi=150, bbox_inches='tight')
    print(f"Bar chart saved to: {bar_path}")
    plt.show()

print("Visualization functions defined.")

## 9. Auto-Generate Report

In [ ]:
def generate_pdf_report(metrics_df: pd.DataFrame, analysis_results: Dict,
                        output_path: Path = None):
    """
    Generate PDF summary report.
    """
    try:
        from fpdf import FPDF
    except ImportError:
        print("FPDF not installed. Generating text report instead.")
        generate_text_report(metrics_df, analysis_results, output_path)
        return

    if output_path is None:
        output_path = EXPERIMENTS_DIR / "report.pdf"

    pdf = FPDF()
    pdf.add_page()

    # Title
    pdf.set_font('Arial', 'B', 16)
    pdf.cell(0, 10, 'GACS Experiment Report', ln=True, align='C')
    pdf.set_font('Arial', '', 10)
    pdf.cell(0, 10, f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}', ln=True, align='C')
    pdf.ln(10)

    # Summary
    pdf.set_font('Arial', 'B', 12)
    pdf.cell(0, 10, 'Summary', ln=True)
    pdf.set_font('Arial', '', 10)
    pdf.cell(0, 8, f"GACS videos: {analysis_results['sample_sizes']['gacs']}", ln=True)
    pdf.cell(0, 8, f"Baseline videos: {analysis_results['sample_sizes']['baseline']}", ln=True)
    pdf.ln(5)

    # Results table
    pdf.set_font('Arial', 'B', 12)
    pdf.cell(0, 10, 'Statistical Comparison', ln=True)
    pdf.set_font('Arial', '', 9)

    # Table header
    pdf.set_fill_color(200, 200, 200)
    pdf.cell(40, 8, 'Metric', border=1, fill=True)
    pdf.cell(30, 8, 'GACS Mean', border=1, fill=True)
    pdf.cell(30, 8, 'Baseline', border=1, fill=True)
    pdf.cell(25, 8, 'Lift', border=1, fill=True)
    pdf.cell(25, 8, 'p-value', border=1, fill=True)
    pdf.cell(20, 8, 'Sig', border=1, fill=True, ln=True)

    # Table rows
    for metric, data in analysis_results['metrics'].items():
        sig = '*' if data['significant'] else ''
        pdf.cell(40, 7, metric, border=1)
        pdf.cell(30, 7, f"{data['gacs_mean']:.2f}", border=1)
        pdf.cell(30, 7, f"{data['baseline_mean']:.2f}", border=1)
        pdf.cell(25, 7, f"{data['lift_percent']:.1f}%", border=1)
        pdf.cell(25, 7, f"{data['p_value']:.4f}", border=1)
        pdf.cell(20, 7, sig, border=1, ln=True)

    pdf.ln(5)
    pdf.set_font('Arial', 'I', 8)
    pdf.cell(0, 8, '* indicates statistical significance (p < 0.05)', ln=True)

    pdf.output(str(output_path))
    print(f"PDF report saved to: {output_path}")

def generate_text_report(metrics_df: pd.DataFrame, analysis_results: Dict,
                         output_path: Path = None):
    """
    Generate text summary report.
    """
    if output_path is None:
        output_path = EXPERIMENTS_DIR / "report.txt"

    lines = [
        "GACS Experiment Report",
        "=" * 60,
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
        "",
        "Summary",
        "-" * 40,
        f"GACS videos: {analysis_results['sample_sizes']['gacs']}",
        f"Baseline videos: {analysis_results['sample_sizes']['baseline']}",
        "",
        "Statistical Comparison",
        "-" * 40,
        f"{'Metric':<25} {'GACS':>10} {'Baseline':>10} {'Lift':>10} {'p-value':>10}",
        "-" * 65
    ]

    for metric, data in analysis_results['metrics'].items():
        sig = "*" if data['significant'] else ""
        lines.append(
            f"{metric:<25} {data['gacs_mean']:>10.2f} {data['baseline_mean']:>10.2f} "
            f"{data['lift_percent']:>9.1f}% {data['p_value']:>10.4f} {sig}"
        )

    lines.extend([
        "-" * 65,
        "* indicates statistical significance (p < 0.05)"
    ])

    with open(output_path, 'w') as f:
        f.write('\n'.join(lines))

    print(f"Text report saved to: {output_path}")

print("Report generation functions defined.")

## 10. Run Experiment

In [ ]:
# Authenticate (run once)
# youtube, youtube_analytics = get_authenticated_services()

In [ ]:
# Upload all videos (run once)
# upload_results = upload_all_videos(youtube, alternate=True)
# display(upload_results)

In [ ]:
# Collect metrics (run daily)
# metrics_df = collect_all_metrics(youtube, youtube_analytics)
# display(metrics_df)

In [ ]:
# Load existing metrics for analysis
metrics_path = EXPERIMENTS_DIR / "metrics.csv"
if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
    print(f"Loaded {len(metrics_df)} metric records")
    display(metrics_df.head())
else:
    print("No metrics found. Collect metrics first.")
    metrics_df = None

In [ ]:
# Analyze results
if metrics_df is not None and len(metrics_df) > 0:
    analysis_results = analyze_experiment_results(metrics_df)
    print_analysis_report(analysis_results)

In [ ]:
# Create visualizations
if metrics_df is not None and len(metrics_df) > 0:
    create_comparison_plots(metrics_df)

In [ ]:
# Generate report
if metrics_df is not None and len(metrics_df) > 0:
    generate_pdf_report(metrics_df, analysis_results)
    generate_text_report(metrics_df, analysis_results)

## 11. Summary

This notebook provides the complete YouTube experiment pipeline:

**Workflow:**
1. `get_authenticated_services()` - Authenticate with YouTube API
2. `upload_all_videos()` - Upload GACS and baseline videos (alternating)
3. `collect_all_metrics()` - Collect performance metrics (run every 24 hours)
4. `analyze_experiment_results()` - Perform t-test comparison
5. `create_comparison_plots()` - Generate visualizations
6. `generate_pdf_report()` - Export PDF summary

**Metrics Collected:**
- `impressions` - Number of times video was shown
- `ctr` - Click-through rate
- `views` - Total views
- `watchTime` - Total watch time (minutes)
- `averageViewDuration` - Average view duration (seconds)
- `averageViewPercentage` - Average retention
- `likes`, `comments`, `shares`
- `subscribersGained`, `subscribersLost`

**Output Files:**
- `data/experiments/youtube_map.json` - YouTube ID mapping
- `data/experiments/metrics.csv` - Time-series metrics
- `data/experiments/comparison_boxplots.png` - Box plots
- `data/experiments/comparison_bars.png` - Bar charts
- `data/experiments/report.pdf` - PDF summary
- `data/experiments/report.txt` - Text summary

**Important Notes:**
- YouTube Analytics data may take 24-48 hours to appear
- Run metrics collection daily for best results
- Wait at least 7 days before final comparison